#### Objetivo: Reporte de rentabilidad por categoría
Se requiere conocer el Margen de Ganancia Neto real por categoría de producto tras descontar costos de producción, impuestos y fletes. Se aplica un prorrateo estimado a nivel de línea sobre los costos logísticos de flete (5%) e impuestos (8%) para evaluar qué categorías de producto sostienen económicamente el negocio.


In [24]:
import sqlite3
import pandas as pd
from IPython.display import display, Markdown
import plotly.express as px
import plotly.graph_objects as go
import dash
from dash import html, dcc
from dash.dependencies import Input, Output


conn = sqlite3.connect("data/AdventureWorks.db")

query_category = """
WITH LinearTransform AS (
    SELECT sod.SalesOrderID,
        sod.ProductID,
        p.Name AS Product,
        cat.Name AS Category,
        sod.OrderQty AS Quantity,
        sod.LineTotal AS GrossIncome,
        (sod.OrderQty * p.StandardCost) AS ProductionCost,
        (sod.LineTotal * 0.05) AS FregihtCost,
        (sod.LineTotal * 0.08) AS TaxCost
    FROM SalesOrderDetail sod
    JOIN Product p ON sod.ProductID = p.ProductID
    JOIN ProductCategory cat ON p.ProductCategoryID = cat.ProductCategoryID
)
SELECT Category,
    SUM(Quantity) AS TotalQuantity,
    ROUND(SUM(GrossIncome), 2) AS TotalGrossIncome,
    ROUND(SUM(ProductionCost), 2) AS TotalProductionCost,
    ROUND(SUM(FregihtCost), 2) AS TotalFregihtCost,
    ROUND(SUM(TaxCost), 2) AS TotalTaxCost,
    ROUND(SUM(GrossIncome - ProductionCost - FregihtCost - TaxCost), 2) AS TotalNetProfit,
    ROUND((SUM(GrossIncome - ProductionCost - FregihtCost - TaxCost) / SUM(GrossIncome)) * 100, 2) AS NetMarginPercentage
FROM LinearTransform
GROUP BY Category
ORDER BY TotalNetProfit DESC;
"""

df_category = pd.read_sql_query(query_category, conn)

display(Markdown("### 1er Dataframe: categorías de productos con su rentabilidad"))
display(df_category)

### 1er Dataframe: categorías de productos con su rentabilidad

,Category,TotalQuantity,TotalGrossIncome,TotalProductionCost,TotalFregihtCost,TotalTaxCost,TotalNetProfit,NetMarginPercentage
0,Vests,121,4309.90,2873.63,215.50,344.79,875.99,20.32
1,Shorts,80,3299.80,2094.10,164.99,263.98,776.73,23.54
2,Helmets,124,2523.88,1622.70,126.19,201.91,573.08,22.71
3,Bike Racks,32,2304.00,1436.16,115.20,184.32,568.32,24.67
4,Cranksets,22,3968.87,2936.96,198.44,317.51,515.95,13.00
5,Hydration Packs,50,1649.70,1028.32,82.49,131.98,406.92,24.67
6,Pedals,84,2996.50,2217.41,149.82,239.72,389.54,13.00
7,Gloves,57,837.56,522.08,41.88,67.00,206.60,24.67
8,Bottom Brackets,22,1320.17,976.93,66.01,105.61,171.62,13.00
9,Derailleurs,21,1296.63,959.51,64.83,103.73,168.56,13.00


#### Objetivo: Desempeño financiero por territorio y canal de venta
Se procesa la distribución geográfica de los ingresos brutos y los márgenes netos de la empresa. Al cruzar los registros de facturación con las regiones y los canales de distribución (Ventas en Línea vs. Tiendas Físicas) se identifica qué mercados internacionales y qué canales comerciales presentan la mayor eficiencia operativa y rentabilidad para el negocio.


In [25]:
query_sales_channel = """
WITH TerritoryTransform AS (
    SELECT
        CASE
            WHEN soh.SalesOrderID % 2 = 0 THEN 'E-Commerce'
            ELSE 'Retail Store'
        END AS SalesChannel,
        IFNULL(a.CountryRegion, 'United States') AS Country,
        soh.SubTotal AS Sales,
        (soh.Freight + soh.TaxAmt) AS MonthlyOperatingCosts
    FROM SalesOrderHeader soh
    LEFT JOIN Address a ON soh.ShipToAddressID = a.AddressID
)
SELECT SalesChannel,
    Country,
    COUNT(*) AS TotalOrders,
    ROUND(SUM(Sales), 2) AS MonthlySales,
    ROUND(SUM(MonthlyOperatingCosts), 2) AS MonthlyCosts,
    ROUND(SUM(Sales - MonthlyOperatingCosts), 2) AS NetProfit,
    ROUND((SUM(Sales - MonthlyOperatingCosts) / SUM(Sales)) * 100, 2) AS NetMarginPercentage
FROM TerritoryTransform
GROUP BY SalesChannel, Country
ORDER BY NetProfit DESC;
"""

df_sales_channel = pd.read_sql_query(query_sales_channel, conn)

display(Markdown("### 2do Dataframe: canal de ventas y territorio"))
display(df_sales_channel)

### 2do Dataframe: canal de ventas y territorio

,SalesChannel,Country,TotalOrders,MonthlySales,MonthlyCosts,NetProfit,NetMarginPercentage
0,E-Commerce,United Kingdom,9,436399.80,45821.98,390577.82,89.5
1,E-Commerce,United States,8,193163.98,20282.22,172881.76,89.5
2,Retail Store,United States,10,154172.70,16188.13,137984.57,89.5
3,Retail Store,United Kingdom,5,81696.63,8578.15,73118.48,89.5


#### Objetivo: Selección de clientes de alto valor
Se agrupan las ventas a nivel individual para identificar a los compradores VIP de la plataforma, consolidando métricas clave como el volumen total de transacciones por cliente (`TotalOrders`), el gasto acumulado (`TotalSpend`) y el valor del ticket promedio (`AverageOrderValue`), para luego seleccionar a los 10 con mayor gasto de compra

In [26]:
query_customers = """
SELECT CustomerID,
    COUNT(SalesOrderID) AS TotalOrders,
    ROUND(SUM(SubTotal), 2) AS TotalSpend,
    ROUND(AVG(SubTotal), 2) AS AverageOrderValue,
    CASE
        WHEN SalesOrderID % 2 = 0 THEN 'E-Commerce'
        ELSE 'Retail Store'
    END AS SalesChannel
FROM SalesOrderHeader
GROUP BY CustomerID
ORDER BY TotalSpend DESC
LIMIT 10;
"""

df_customers = pd.read_sql_query(query_customers, conn)
conn.close()

display(Markdown("### 3er Dataframe: clientes de alto valor"))
display(df_customers)

### 3er Dataframe: clientes de alto valor

,CustomerID,TotalOrders,TotalSpend,AverageOrderValue,SalesChannel
0,29736,1,108561.83,108561.83,E-Commerce
1,30050,1,98278.69,98278.69,E-Commerce
2,29546,1,88812.86,88812.86,E-Commerce
3,29957,1,83858.43,83858.43,Retail Store
4,29796,1,78029.69,78029.69,Retail Store
5,29929,1,74058.81,74058.81,E-Commerce
6,29932,1,63980.99,63980.99,E-Commerce
7,29660,1,57634.63,57634.63,E-Commerce
8,29938,1,41622.05,41622.05,Retail Store
9,29485,1,39785.33,39785.33,E-Commerce


#### Herramienta interactiva que monitorea los KPIs mensuales de las distintas entidades procesadas

In [ ]:
total_company_revenue = df_category["TotalGrossIncome"].sum()
total_company_net_profit = df_category["TotalNetProfit"].sum()
company_wide_net_margin_baseline = (total_company_net_profit / total_company_revenue) * 100

top_volume_row = df_category.sort_values(by="TotalSales", ascending=False).iloc[0]
global_total_units = df_category["TotalSales"].sum()
top_volume_market_share = (top_volume_row["TotalSales"] / global_total_units) * 100
top_revenue_row = df_category.sort_values(by="TotalGrossIncome", ascending=False).iloc[0]
top_efficiency_row = df_category.sort_values(by="NetMarginPercentage", ascending=False).iloc[0]

total_company_orders = df_sales_channel["TotalOrders"].sum()
total_company_costs = df_sales_channel["MonthlyCosts"].sum()
company_cost_per_order_baseline = total_company_costs / total_company_orders

top_regional_volume = df_sales_channel.sort_values(by="TotalOrders", ascending=False).iloc
top_regional_profit = df_sales_channel.sort_values(by="NetProfit", ascending=False)
df_sales_channel["CostPerOrder"] = df_sales_channel["MonthlyCosts"] / df_sales_channel["TotalOrders"]
top_cost_efficiency = df_sales_channel.sort_values(by="CostPerOrder", ascending=True).iloc

total_spend_global = df_customers["TotalSpend"].sum()
total_orders_global = df_customers["TotalOrders"].sum()
company_wide_aov_baseline = total_spend_global / total_orders_global

top_monetary_customer = df_customers.sort_values(by="TotalSpend", ascending=False).iloc[0]
top_frequency_customer = df_customers.sort_values(by="TotalOrders", ascending=False).iloc[0]
top_ticket_customer = df_customers.sort_values(by="AverageOrderValue", ascending=False).iloc[0]

app = dash.Dash(__name__)

app.layout = html.Div(id="body",children=[
    html.H1("AdventureWorks Analytics: Panel de control financiero y omnicanal mensual", className="e3_title", style={"margin-bottom":"50px"}),
    html.Div(id="dropdown_div", className="e3_dropdown_div", children=[
            dcc.Dropdown(id="dropdown", className="e3_dropdown",
                        options = [
                            {"label":"Categorías","value":"Category"},
                            {"label":"Canal de ventas","value":"SalesChannel"},
                            {"label":"Clientes","value":"CustomerID"}
                        ],
                        value="name",
                        multi=False,
                        clearable=False)
    ]),
    dcc.Graph(id="figure-1", figure={}),
    html.H2("Palancas de negocio", className="e3_title"),
    html.Div(className="e3_container", children=[
        html.Div(id="data_1", className="e3_children", children=[
            html.H2("Categorías", style={"font-size":"1.15em","font-family":"sans-serif"}),
            html.P(f"Promedio (Margen Neto): {company_wide_net_margin_baseline}%", className="e3_mean"),
            html.Ul(className="e3_ul", children=[
                html.Li(f"Volumen de ventas ({top_volume_row["Category"]}): {top_volume_market_share["TotalSales"]}", className="e3_list"),
                html.Li(f"Ingresos Brutos ({top_revenue_row["Category"]}): ${top_revenue_row["TotalGrossIncome"]}", className="e3_list"),
                html.Li(f"Margen Neto ({top_efficiency_row["Category"]}): {top_efficiency_row["NetMarginPercentage"]}%", className="e3_list")
            ])
        ]),
        html.Div(id="data_2", className="e3_children", children=[
            html.H2("Canal de ventas", style={"font-size":"1.15em","font-family":"sans-serif"}),
            html.P(f"Promedio (CPO): ${company_cost_per_order_baseline}", className="e3_mean"),
            html.Ul(className="e3_ul", children=[
                html.Li(f"Volumen de órdenes ({top_regional_volume["SalesChannel"]}): {top_regional_volume["TotalOrders"]}", className="e3_list"),
                html.Li(f"Ganancia Neta ({top_regional_profit["SalesChannel"]}): ${top_regional_profit["NetProfit"]}", className="e3_list"),
                html.Li(f"Eficiencia de costos ({top_cost_efficiency["SalesChannel"]}): {top_cost_efficiency["CostPerOrder"]}", className="e3_list")
            ])
        ]),
        html.Div(id="data_3", className="e3_children", children=[
            html.H2("Clientes", style={"font-size":"1.15em","font-family":"sans-serif"}),
            html.P(f"Promedio (AOV): ${company_wide_aov_baseline}", className="e3_mean"),
            html.Ul(className="e3_ul", children=[
                html.Li(f"Gasto total ({top_monetary_customer["CustomerID"]}): ${top_monetary_customer["TotalSpend"]}", className="e3_list"),
                html.Li(f"Frecuencia ({top_frequency_customer["CustomerID"]}): {top_frequency_customer["TotalOrders"]}", className="e3_list"),
                html.Li(f"Valor promedio de órden ({top_ticket_customer["CustomerID"]}): ${top_ticket_customer["AverageOrderValue"]}", className="e3_list")
            ])
        ])
    ]),
    html.Div(id="dropdown_2_div", className="e3_div_dropdown", children=[
        dcc.Dropdown(id="dropdown_category", className="e3_dropdown",
                    options=df_category["Category"].tolist(),
                    value=df_category["Category"].iloc[0],
                    multi=False,
                    clearable=False),
        dcc.Dropdown(id="dropdown_sales_channel", className="e3_dropdown",
                    options=df_sales_channel["SalesChannel"].tolist(),
                    value=df_sales_channel["SalesChannel"].iloc[0],
                    multi=False,
                    clearable=False),
        dcc.Dropdown(id="dropdown_customer", className="e3_dropdown",
                    options=df_customers["CustomerID"].tolist(),
                    value=df_customers["CustomerID"].iloc[0],
                    multi=False,
                    clearable=False)
    ]),
    dcc.Graph(id="figure-2",figure={})
])

@app.callback(
    [Output(component_id="figure-1", component_property="figure"),
    Output(component_id="dropdown_category", component_property="style"),
    Output(component_id="dropdown_sales_channel", component_property="style"),
    Output(component_id="dropdown_customer", component_property="style"),
    Output(component_id="figure-2", component_property="figure")],
    [Input(component_id="dropdown", component_property="value"),
    Input(component_id="dropdown_category", component_property="value"),
    Input(component_id="dropdown_sales_channel", component_property="value"),
    Input(component_id="dropdown_customer", component_property="value")]
)


def update_dashboard(slct_data, slct_category, slct_sales_channel, slct_customer):

    category_style = {"position":"absolute","top":"0","left":"0"}
    sales_channel_style = {"position":"absolute","top":"0","left":"0"}
    customer_style = {"position":"absolute","top":"0","left":"0"}

    figure_1 = go.Figure()
    figure_2 = go.Figure()

    if slct_data == "Category":

        category_style["zIndex"] = 5

        figure_1 = px.bar(
            df_category,
            x="Category",
            y=["TotalGrossIncome", "TotalProfitNet"],
            barmode="group",
            title="Ingreso Bruto vs Ganancia Neta",
            labels={"value": "Amount ($)", "variable": "Métrica Financiera", "Category": "Categoría"},
            template="plotly_dark"
        )

        df_filtered = df_category[df_category["Category"] == slct_category]
        row = df_filtered.iloc[0]

        figure_2 = px.pie(
              names=["Costo de Producción", "Costo de Flete", "Costo de Impuesto", "Ganancia Neta"],
              values=[row["TotalProductionCost"], row["TotalFreightCost"], row["TotalTaxCost"], row["TotalNetProfit"]],
              title=f"Descomposición Financiera: {slct_category} (Margen Neto de Junio: {row["NetMarginPercentage"]:.1f}%)",
              template="plotly_dark",
              hole=0.4,
              color_discrete_sequence=px.colors.qualitative.Pastel
        )

    elif slct_data == "SalesChannel":

        sales_channel_style["zIndex"] = 5

        figure_1 = px.bar(
            df_sales_channel,
            x="SalesChannel",
            y="MonthlySales",
            color="Country",
            barmode="group",
            title="Ventas Mensuales por Canal y Territorio",
            labels={"MonthlySales": "Ventas ($)", "SalesChannel": "Canal de Ventas"},
            template="plotly_dark"
        )

        df_filtered = df_sales_channel[df_sales_channel["SalesChannel"] == slct_sales_channel]

        figure_2 = px.bar(
            df_filtered,
            x="Country",
            y="NetMarginPercentage",
            title=f"Eficiencia del Margen Neto por País para el Canal: {slct_sales_channel}",
            labels={"NetMarginPercentage": "Margen Neto (%)", "Country": "País"},
            template="plotly_dark"
        ).update_traces(marker_color="#34d399")

    elif slct_data == "CustomerID":

        customer_style["zIndex"] = 5

        figure_1 = px.bar(
            df_customers,
            x="CustomerID",
            y="TotalSpend",
            color="SalesChannel",
            title="Top 10 Clientes VIP por Gasto Total Acumulado",
            labels={"TotalSpend": "Gasto Total ($)", "CustomerID": "ID"},
            template="plotly_dark"
        )

        df_filtered = df_customers[df_customers["CustomerID"] == int(slct_customer)]
        row = df_filtered.iloc[0]

        figure_2 = px.bar(
              x=["Gasto Total", "Valor de Ticket Promedio (AOV)"],
              y= [row["TotalSpend"], row["AverageOrderValue"]],
              title=f"Perfil Financiero Individual del Cliente ID: {slct_customer}",
              labels={"x": "Métrica Comercial", "y": "Valor ($)"},
              template="plotly_dark"
        ).update_traces(marker_color="#10b981")

    else:
        figure_2 = go.Figure()


    return figure_1, category_style, sales_channel_style, customer_style, figure_2


if __name__ == "__main__":
    app.run(debug=False)